# Sampling

We want to collect reviews wrote by active users, meaning :
- Users with more than 20 reviews
- A sample of 50,000 users among them 
- For active each user, collect all his reviews

| Métrique | Valeur |
|----------|--------|
| Reviews totales | ~27 millions |
| Utilisateurs uniques | 10 297 355 |
| Utilisateurs actifs (≥ 20 reviews) | 137 305 (1,33 % des utilisateurs) |
| Utilisateurs échantillonnés | 50 000 (36,4 % des actifs) |


| Itération | Outil | Reviews | Users | Volumétrie OK ? | Structure préservée ? | Bug notable |
|-----------|-------|---------|-------|-----------------|-----------------------|-------------|
| 1 | Streaming Python | 2 442 098 | 50 000 | Légèrement > 2M | Oui (verbatim) | — |
| 2 | Pandas chunked | ~2,4M | 50 000 | Légèrement > 2M | Oui (tous champs) | — |
| 3 | Polars | 2 425 431 | 50 000 | Légèrement > 2M | Oui | Filtre temporel = 0 (ms vs s) |
| 4 | Dask CPU | 2 415 706 | 50 000 | Légèrement > 2M | 9 colonnes sélectionnées | — |
| 5A | cuDF row-groups | ~2,44M | 50 000 | Légèrement > 2M | Oui (tous champs) | — |
| 5B | cuDF byte-range | 2 442 267 | 50 000 | Légèrement > 2M | Oui | — |
| 5C | cuDF + RMM | 2 442 267 | 50 000 | Légèrement > 2M | Oui | — |
| 5D | cuDF temporel | 200 000 | N/A | Non (trop faible) | Profils brisés | Timestamp ms/s + volume |
| 6 | Dask + cuDF | 2 420 879 | 50 000 | Légèrement > 2M | 8 colonnes sélectionnées | — |
| 7 | PySpark | Non exécuté | — | — | — | Sampling approximatif |
| 8 | DuckDB | 2 429 097 | 50 000 | Légèrement > 2M | Oui (tous champs) | Filtre temporel (ms vs s) |

### First things first ###
Convert the JSONL to Parquet first. Every subsequent read will be 10-50x faster:

In [3]:
import polars as pl
import duckdb

# With Polars:
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

# # With DuckDB:
# duckdb.sql("""
#     COPY (SELECT * FROM read_json_auto('data/Books.jsonl', format='newline_delimited'))
#     TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
# """)

# DuckDB handles messy JSONL flawlessly and writes Parquet very fast
con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~5-10 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books-duckdb.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

Converting JSONL → Parquet (this takes ~5-10 min)...


RuntimeError: Query interrupted

#### Représentativité
- Cette cellule est une étape de prétraitement pur : la conversion JSONL → Parquet ne modifie aucune donnée et ne filtre rien. L'intégralité des ~27M reviews est fidèlement transcrite.
#### Volumétrie cible
- Sans impact direct. Le fichier Parquet contient exactement le même nombre de lignes que le JSONL source.
#### Préservation de la structure
- Le format Parquet préserve tous les champs et types du JSONL original. De plus, il ajoute une structuration en colonnes avec compression et row groups (ROW_GROUP_SIZE=1_000_000), ce qui permet aux outils en aval (Dask, cuDF, DuckDB) de faire du predicate pushdown et du column pruning sans charger le fichier entier — un avantage structurel majeur pour les itérations suivantes.

### First Sampling iteration Streaming Python ###

The simplest and most memory-efficient. Uses almost zero RAM beyond the dictionaries.


In [ ]:
import json
import random
from collections import Counter
from datetime import datetime

DATA_PATH = "data/Books.jsonl"
OUTPUT_PATH = "sample-streaming-python/sampled_reviews.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user ──────────────────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

with open(DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        user_counts[record["user_id"]] += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines...")

print(f"Total unique users: {len(user_counts):,}")

# ── Filter active users (>= 20 reviews) ────────────────────────
active_users = [uid for uid, count in user_counts.items() if count >= MIN_REVIEWS]
print(f"Active users (>= {MIN_REVIEWS} reviews): {len(active_users):,}")

# ── Sample 50,000 users ────────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
print(f"Sampled users: {len(sampled_users):,}")

del user_counts, active_users  # free memory

# ── Pass 2: Extract reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews for sampled users...")
total_extracted = 0

with open(DATA_PATH, "r") as fin, open(OUTPUT_PATH, "w") as fout:
    for i, line in enumerate(fin):
        record = json.loads(line)
        if record["user_id"] in sampled_users:
            fout.write(line)
            total_extracted += 1
        if i % 5_000_000 == 0:
            print(f"  Processed {i:,} lines, extracted {total_extracted:,}")

print(f"Total extracted reviews: {total_extracted:,}")
    


Pass 1: Counting reviews per user...
  Processed 0 lines...
  Processed 5,000,000 lines...
  Processed 10,000,000 lines...
  Processed 15,000,000 lines...
  Processed 20,000,000 lines...
  Processed 25,000,000 lines...
Total unique users: 10,297,355
Active users (>= 20 reviews): 137,305
Sampled users: 50,000
Pass 2: Extracting reviews for sampled users...
  Processed 0 lines, extracted 0
  Processed 5,000,000 lines, extracted 745,349
  Processed 10,000,000 lines, extracted 1,323,541
  Processed 15,000,000 lines, extracted 1,852,537
  Processed 20,000,000 lines, extracted 2,204,841
  Processed 25,000,000 lines, extracted 2,428,722
Total extracted reviews: 2,442,098


Résultat : 2 442 098 reviews, 50 000 utilisateurs
#### Représentativité
- Passe 1 : parcours exhaustif des ~27M lignes pour compter les reviews par utilisateur. Aucun utilisateur n'est oublié.
- Filtrage : seuil ≥ 20 reviews → 137 305 utilisateurs actifs identifiés.
- Tirage : random.sample avec seed=42 sur les 137 305 actifs → 50 000 utilisateurs tirés uniformément. La probabilité de sélection est identique pour chaque utilisateur actif (~36,4 %), sans biais lié à la position dans le fichier ou au nombre de reviews.
- Passe 2 : collecte exhaustive de toutes les reviews des utilisateurs sélectionnés. Le set lookup (in sampled_users) est O(1) et ne discrimine pas selon l'ordre du fichier.
#### Volumétrie cible
- 2 442 098 reviews — légèrement au-dessus de 2M. La distribution long-tail des reviews par utilisateur (certains "super-reviewers" ont des centaines de reviews) tire le total vers le haut. Pour rester strictement dans [500K, 2M], on pourrait réduire NUM_USERS à ~42 000.
#### Préservation de la structure
- Les lignes JSONL sont copiées verbatim (fout.write(line)) — aucune transformation, aucune perte de champ. Chaque review conserve son user_id, parent_asin, rating, timestamp, text, title, helpful_vote, verified_purchase, images, etc.
- Le profil complet de chaque utilisateur est préservé : toutes ses reviews, toutes ses notes, toute sa chronologie.

### Second Sampling iteration Using Pandas with Chunked Reading ###

In [ ]:
import pandas as pd
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 500_000  # rows per chunk
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user (chunked) ───────────────────
print("Pass 1: Counting reviews per user...")
user_counts = Counter()

reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    counts = chunk["user_id"].value_counts()
    user_counts.update(counts.to_dict())
    print(f"  Chunk {i}: {len(chunk):,} rows processed")

# ── Filter & sample active users ───────────────────────────────
active_users = [uid for uid, c in user_counts.items() if c >= MIN_REVIEWS]
print(f"Active users: {len(active_users):,}")

random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del user_counts, active_users

# ── Pass 2: Collect reviews for sampled users ──────────────────
print("Pass 2: Extracting reviews...")
chunks = []
reader = pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE,
                       dtype={"user_id": str, "parent_asin": str, "rating": float})

for i, chunk in enumerate(reader):
    filtered = chunk[chunk["user_id"].isin(sampled_users)]
    if len(filtered) > 0:
        chunks.append(filtered)
    print(f"  Chunk {i}: kept {len(filtered):,} / {len(chunk):,}")

df = pd.concat(chunks, ignore_index=True)
print(f"Final sample: {len(df):,} reviews from {df['user_id'].nunique():,} users")

# ── Save ────────────────────────────────────────────────────────
df.to_parquet("sample-pandas-with-chunk/sampled_reviews.parquet", index=False)
df.to_json("sample-pandas-with-chunk/sampled_reviews.jsonl", orient="records", lines=True)

Pass 1: Counting reviews per user...
  Chunk 0: 500,000 rows processed
  Chunk 1: 500,000 rows processed
  Chunk 2: 500,000 rows processed
  Chunk 3: 500,000 rows processed
  Chunk 4: 500,000 rows processed
  Chunk 5: 500,000 rows processed
  Chunk 6: 500,000 rows processed
  Chunk 7: 500,000 rows processed
  Chunk 8: 500,000 rows processed
  Chunk 9: 500,000 rows processed
  Chunk 10: 500,000 rows processed
  Chunk 11: 500,000 rows processed
  Chunk 12: 500,000 rows processed
  Chunk 13: 500,000 rows processed
  Chunk 14: 500,000 rows processed
  Chunk 15: 500,000 rows processed
  Chunk 16: 500,000 rows processed
  Chunk 17: 500,000 rows processed
  Chunk 18: 500,000 rows processed
  Chunk 19: 500,000 rows processed
  Chunk 20: 500,000 rows processed
  Chunk 21: 500,000 rows processed
  Chunk 22: 500,000 rows processed
  Chunk 23: 500,000 rows processed
  Chunk 24: 500,000 rows processed
  Chunk 25: 500,000 rows processed
  Chunk 26: 500,000 rows processed
  Chunk 27: 500,000 rows pro

Résultat : mêmes 50 000 utilisateurs, ~2,4M reviews
#### Représentativité
- Strictement identique à l'itération 1 en termes de logique : même seed=42, même seuil ≥ 20, même random.sample. La seule différence est l'outillage (pandas chunks de 500K lignes au lieu d'un parsing JSON ligne par ligne).
- Le chunksize=500_000 ne biaise pas le comptage : les Counter sont accumulés sur tous les chunks.
#### Volumétrie cible
- Le volume final est cohérent avec l'itération 1 (~2,4M). La légère variation potentielle viendrait de différences de parsing de types (dtype explicites), mais le nombre d'utilisateurs actifs est identique (137 305).
#### Préservation de la structure
- L'utilisation de pd.read_json(..., lines=True) charge tous les champs du JSONL dans un DataFrame. Le isin sur user_id ne supprime aucune colonne. La sortie est sauvée en Parquet + JSONL, préservant à la fois un format analytique performant et un format texte lisible.
- Les types sont explicitement contrôlés (user_id: str, rating: float) pour éviter les conversions silencieuses de pandas.

### Third Sampling iteration Using Polars ###

Polars is a Rust-based DataFrame library that is 5-10x faster than pandas for this workload thanks to multi-threaded execution and lazy evaluation.

In [ ]:
import polars as pl
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Polars lazy scan (memory-mapped, multi-threaded) ────────────
# This does NOT load the file into RAM — it creates a query plan
lf = pl.scan_ndjson(
    DATA_PATH,
    ignore_errors=True,
    low_memory=False,
    # Don't specify full schema — let Polars infer with overrides
    schema_overrides={
        "user_id": pl.Utf8,
        "rating": pl.Float64,
        "timestamp": pl.Int64,
    },
)

# ── Step 1: Find active users ──────────────────────────────────
print("Step 1: Identifying active users...")
user_review_counts = (
    lf.select("user_id")           # <── project FIRST, reduces data + schema issues
    .group_by("user_id")
    .agg(pl.len().alias("review_count"))
    .filter(pl.col("review_count") >= MIN_REVIEWS)
    .collect()  # materializes only the aggregation result
)
print(f"Active users (>= {MIN_REVIEWS}): {len(user_review_counts):,}")

# ── Step 2: Sample 50,000 users ────────────────────────────────
random.seed(SEED)
all_active = user_review_counts["user_id"].to_list()
sampled_users = random.sample(all_active, min(NUM_USERS, len(all_active)))
sampled_set = pl.Series("user_id", sampled_users)

# ── Step 3: Filter reviews ─────────────────────────────────────
print("Step 2: Filtering reviews for sampled users...")
sampled_df = (
    lf.filter(pl.col("user_id").is_in(sampled_set))
    .collect()
)
print(f"Sampled: {len(sampled_df):,} reviews, {sampled_df['user_id'].n_unique():,} users")

# ── Optional temporal filter (combine strategies) ───────────────
# Unix timestamp for Jan 1, 2020 = 1577836800
# Unix timestamp for Dec 31, 2023 = 1703980800
sampled_temporal = sampled_df.filter(
    (pl.col("timestamp") >= 1577836800) & (pl.col("timestamp") <= 1703980800)
)
print(f"After temporal filter (2020-2023): {len(sampled_temporal):,} reviews")

# ── Save ────────────────────────────────────────────────────────
sampled_df.write_parquet("sample-polars/sampled_reviews.parquet")
sampled_df.write_json("sample-polars/sampled_reviews.jsonl")

Step 1: Identifying active users...
Active users (>= 20): 137,305
Step 2: Filtering reviews for sampled users...


/tmp/ipykernel_3232/3053399840.py:44: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


Sampled: 2,425,431 reviews, 50,000 users
After temporal filter (2020-2023): 0 reviews


Résultat : 2 425 431 reviews, 50 000 utilisateurs
#### Représentativité
- Même logique d'échantillonnage (seed=42, ≥ 20, random.sample de 50K). Le nombre d'utilisateurs actifs retrouvé est identique : 137 305.
- La légère différence de volume (2 425 431 vs 2 442 098) peut s'expliquer par le ignore_errors=True dans scan_ndjson, qui saute silencieusement les lignes mal formées. Certaines reviews parsées par le streaming Python sont potentiellement rejetées par Polars. C'est un compromis : on perd quelques lignes edge-case mais on gagne en robustesse.
#### Volumétrie cible
- 2 425 431 reviews — toujours dans la fourchette haute. Un filtre temporel est testé (2020–2023) mais retourne 0 reviews car les timestamps sont en millisecondes (ex: 850796830000) alors que les bornes sont en secondes. C'est un bug dans cette cellule, pas un choix de design.
#### Préservation de la structure
- Polars scan_ndjson + collect() charge toutes les colonnes. Le schema_overrides force les bons types sans supprimer de champs.
- Le group_by("user_id").agg(pl.len()) est purement analytique et ne modifie pas le DataFrame source.
- Le filtre final (is_in(sampled_set)) conserve toutes les colonnes de chaque review.

### Fourth Sampling iteration using Dask ###

Dask extends pandas to larger-than-memory datasets with lazy parallel execution.

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import random

DATA_PATH = "data/Books-polars.parquet"  # USE PARQUET, not JSONL!
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Configure Dask with explicit memory limits ─────────────────
# This prevents workers from consuming all RAM and crashing the machine
cluster = LocalCluster(
    n_workers=4,              # fewer workers = less concurrent memory
    threads_per_worker=6,
    memory_limit="12GB",      # per-worker limit (4 × 12 = 48GB ceiling)
)
client = Client(cluster)
print(client.dashboard_link)  # monitor at http://localhost:8787

# ── Step 1: Count reviews per user (project ONLY user_id) ──────
print("Counting reviews per user...")
ddf = dd.read_parquet(DATA_PATH, columns=["user_id"])  # only 1 column!
user_counts = ddf.groupby("user_id").size().compute()  # small Series result

active_users = user_counts[user_counts >= MIN_REVIEWS].index.tolist()
print(f"Active users: {len(active_users):,}")

del ddf, user_counts  # free Dask graph references

# ── Step 2: Sample users (CPU, trivial) ────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del active_users

# ── Step 3: Filter reviews and write directly to disk ──────────
print("Filtering reviews...")
NEEDED_COLS = ["user_id", "parent_asin", "rating", "timestamp",
               "title", "text", "helpful_vote", "verified_purchase", "asin"]
ddf_full = dd.read_parquet(DATA_PATH, columns=NEEDED_COLS)
sampled_ddf = ddf_full[ddf_full["user_id"].isin(sampled_users)]

# Write directly to parquet WITHOUT calling .compute()
# This streams partitions to disk one at a time instead of materializing all in RAM
sampled_ddf.to_parquet(
    "sample-dask/sampled_reviews-polars.parquet",
    write_index=False,
    overwrite=True,
)

# If you need a single file or want to also get the count:
result = dd.read_parquet("sample-dask/sampled_reviews-polars.parquet")
print(f"Sampled: {len(result):,} reviews")

# ── Cleanup ────────────────────────────────────────────────────
client.close()
cluster.close()

import pandas as pd

# Read back the already-filtered, much smaller parquet
sampled_df = pd.read_parquet("sample-dask/sampled_reviews-polars.parquet")
sampled_df.to_parquet("sample-dask/sampled_reviews-polars-single-file.parquet")
print(f"Sampled: {len(sampled_df):,} reviews from {sampled_df['user_id'].nunique():,} users")
sampled_df.to_json("sample-dask/sampled_reviews-polars.jsonl", orient="records", lines=True)

/usr/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43171 instead
  warnings.warn(


http://127.0.0.1:43171/status
Counting reviews per user...
Active users: 137,305
Filtering reviews...
Sampled: 2,415,706 reviews
Sampled: 2,415,706 reviews from 50,000 users


Résultat : 2 415 706 reviews, 50 000 utilisateurs
#### Représentativité
- Lit le fichier Parquet (converti en Cell 2 par Polars). Même seed, même seuil, même random.sample.
- Dask utilise un LocalCluster avec 4 workers × 6 threads. Le groupby("user_id").size() est parallélisé sur les partitions Parquet puis réduit. Le résultat (137 305 actifs) est cohérent.
- La variation de volume (2 415 706 vs 2 442 098) vient probablement du fait que le Parquet source est celui généré par Polars (Books-polars.parquet), qui peut avoir des différences de parsing par rapport au JSONL brut.
#### Volumétrie cible
- 2 415 706 reviews — toujours au-dessus de 2M. Remarque : seules 9 colonnes sont sélectionnées (NEEDED_COLS), ce qui est un choix de projection réduisant l'empreinte mémoire sans affecter le nombre de lignes.
#### Préservation de la structure
- Le NEEDED_COLS sélectionne explicitement : user_id, parent_asin, rating, timestamp, title, text, helpful_vote, verified_purchase, asin. Ce sont les champs essentiels pour l'analyse. Les champs exclus (ex: images) sont jugés non nécessaires pour le downstream.
- L'écriture via to_parquet(..., overwrite=True) est faite partition par partition sans .compute(), évitant de tout matérialiser en RAM — ce qui préserve la faisabilité du pipeline sur des machines à RAM limitée.

### Fifth Sampling iteration using cuDF / RAPIDS ###

This is the GPU powerhouse. cuDF mirrors the pandas API but runs on NVIDIA GPUs. You need an NVIDIA GPU with sufficient VRAM (ideally 16GB+ for this dataset).

### Itération 5A : cuDF/RAPIDS (GPU, ≥24GB VRAM) ###

If the dataset fits in GPU VRAM (24GB+ GPU), cuDF can read JSONL directly on GPU.

In [ ]:
import cudf
import pyarrow.parquet as pq
import pandas as pd
import random

DATA_PATH = "data/Books-polars.parquet"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ============================================================
# PHASE 1: Find active users and sample 50,000
# (Only loads the user_id column — very light on VRAM)
# ============================================================

print("Phase 1: Counting reviews per user...")
gdf_ids = cudf.read_parquet(DATA_PATH, columns=["user_id"])

user_counts = gdf_ids["user_id"].value_counts().reset_index()
user_counts.columns = ["user_id", "count"]
active = user_counts[user_counts["count"] >= MIN_REVIEWS]
print(f"Active users (>= {MIN_REVIEWS} reviews): {len(active):,}")

# Sample 50,000 users on CPU
random.seed(SEED)
active_list = active["user_id"].to_pandas().tolist()
sampled = random.sample(active_list, min(NUM_USERS, len(active_list)))
sampled_list = list(sampled)
print(f"Sampled users: {len(sampled_list):,}")

# Free GPU memory completely before Phase 2
del gdf_ids, user_counts, active

# ============================================================
# PHASE 2: Extract all reviews for sampled users
# (Reads one row group at a time to avoid VRAM overflow)
# ============================================================

print("\nPhase 2: Extracting reviews for sampled users...")

pf = pq.ParquetFile(DATA_PATH)
n_row_groups = pf.metadata.num_row_groups
print(f"Parquet has {n_row_groups} row groups to process")

result_chunks = []
total_matched = 0

for rg in range(n_row_groups):
    # Load one row group into GPU (~1M rows, ~1-2 GB VRAM)
    chunk = cudf.read_parquet(DATA_PATH, row_groups=[rg])
    
    # Filter on GPU: keep only rows matching our 50K users
    filtered = chunk[chunk["user_id"].isin(sampled_list)]
    
    matched = len(filtered)
    total_matched += matched
    
    # Move small filtered result to CPU, free GPU memory
    if matched > 0:
        result_chunks.append(filtered.to_pandas())
    
    del chunk, filtered
    print(f"  Row group {rg + 1}/{n_row_groups}: matched {matched:,} rows (total: {total_matched:,})")

# ============================================================
# PHASE 3: Combine results and save
# ============================================================

print("\nPhase 3: Combining and saving...")
df = pd.concat(result_chunks, ignore_index=True)
print(f"Final sample: {len(df):,} reviews from {df['user_id'].nunique():,} users")

# Optional temporal filter (2020-2023)
df_temporal = df[(df["timestamp"] >= 1577836800) & (df["timestamp"] <= 1703980800)]
print(f"After temporal filter (2020-2023): {len(df_temporal):,} reviews")

# Save
df.to_parquet("sample-cudf-rapids-24+/sampled_reviews.parquet", index=False)
print("Saved to sample-cudf-rapids-24+/sampled_reviews.parquet")
df.to_json("sample-cudf-rapids-24+/sampled_reviews.jsonl",orient="records", lines=True)
print("Saved to sample-cudf-rapids-24+/sampled_reviews.jsonl")

Phase 1: Counting reviews per user...
Active users (>= 20 reviews): 137,305
Sampled users: 50,000

Phase 2: Extracting reviews for sampled users...
Parquet has 240 row groups to process
  Row group 1/240: matched 30,481 rows (total: 30,481)
  Row group 2/240: matched 26,172 rows (total: 56,653)
  Row group 3/240: matched 23,162 rows (total: 79,815)
  Row group 4/240: matched 16,972 rows (total: 96,787)
  Row group 5/240: matched 18,092 rows (total: 114,879)
  Row group 6/240: matched 22,846 rows (total: 137,725)
  Row group 7/240: matched 18,682 rows (total: 156,407)
  Row group 8/240: matched 14,575 rows (total: 170,982)
  Row group 9/240: matched 13,767 rows (total: 184,749)
  Row group 10/240: matched 14,876 rows (total: 199,625)
  Row group 11/240: matched 13,893 rows (total: 213,518)
  Row group 12/240: matched 14,344 rows (total: 227,862)
  Row group 13/240: matched 14,252 rows (total: 242,114)
  Row group 14/240: matched 17,361 rows (total: 259,475)
  Row group 15/240: matched 2

Résultat : ~2,44M reviews, 50 000 utilisateurs
#### Représentativité
- Même logique en 2 phases : Phase 1 (comptage sur GPU colonne user_id seule), Phase 2 (filtrage row-group par row-group).
- Le traitement par row groups individuels (240 au total) ne biaise pas l'échantillonnage : chaque row group est filtré avec le même ensemble de 50K user_id.
- Le transfert CPU/GPU est fait sur les résultats filtrés uniquement (filtered.to_pandas()), préservant la fidélité des données.
#### Volumétrie cible
- Le total (visible dans la sortie tronquée) converge vers ~2,44M. C'est cohérent avec les autres itérations utilisant la même logique.
#### Préservation de la structure
- Le cudf.read_parquet(DATA_PATH, row_groups=[rg]) charge toutes les colonnes de chaque row group.
- Le filtre isin(sampled_list) ne touche qu'aux lignes, pas aux colonnes.
- La concaténation finale (pd.concat(result_chunks)) reconstitue un DataFrame complet.

### Itération 5B : cuDF chunked (VRAM limitée) ###

Chunked GPU processing is more memory-efficient for limited VRAM.

In [ ]:
import cudf
import pandas as pd
import random
from collections import Counter

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Pass 1: Count reviews per user (chunked on GPU) ────────────
print("Pass 1: Counting reviews per user on GPU...")
user_counts = Counter()

# Use byte_range to read the file in chunks
import os
file_size = os.path.getsize(DATA_PATH)
CHUNK_BYTES = 400_000_000_000  # 500MB chunks — adjust based on your VRAM

offset = 0
chunk_num = 0
while offset < file_size:
    size = min(CHUNK_BYTES, file_size - offset)
    try:
        chunk = cudf.read_json(
            DATA_PATH,
            lines=True,
            engine="cudf",
            byte_range=(offset, size),  # <── read only a slice of the file
        )
        # Aggregate on GPU, transfer tiny result to CPU
        counts = chunk["user_id"].value_counts()
        cpu_counts = counts.to_pandas()
        user_counts.update(cpu_counts.to_dict())
        
        del chunk, counts, cpu_counts  # free GPU memory immediately
        print(f"  Chunk {chunk_num}: offset={offset:,}, size={size:,}")
    except Exception as e:
        print(f"  Chunk {chunk_num} error (skipping): {e}")
    
    offset += size
    chunk_num += 1

# ── Filter active users ────────────────────────────────────────
active_users = [u for u, c in user_counts.items() if c >= MIN_REVIEWS]
print(f"Active users: {len(active_users):,}")

random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del user_counts, active_users

# ── Pass 2: Filter reviews (chunked on GPU) ────────────────────
print("Pass 2: Extracting reviews on GPU...")
sampled_list = list(sampled_users)  # cudf.isin needs a list
result_chunks = []

offset = 0
while offset < file_size:
    size = min(CHUNK_BYTES, file_size - offset)
    try:
        chunk = cudf.read_json(
            DATA_PATH,
            lines=True,
            engine="cudf",
            byte_range=(offset, size),
        )
        # Filter on GPU, transfer only matching rows to CPU
        filtered = chunk[chunk["user_id"].isin(sampled_list)]
        if len(filtered) > 0:
            result_chunks.append(filtered.to_pandas())
        del chunk, filtered
    except Exception:
        pass

    offset += size

df = pd.concat(result_chunks, ignore_index=True)
print(f"Sampled: {len(df):,} reviews from {df['user_id'].nunique():,} users")

df.to_parquet("sample-cudf-rapids-24-/sampled_reviews.parquet", index=False)
df.to_json("sample-cudf-rapids-24-/sampled_reviews.jsonl", orient="records", lines=True)

Pass 1: Counting reviews per user on GPU...
  Chunk 0: offset=0, size=20,121,186,727
Active users: 137,305
Pass 2: Extracting reviews on GPU...
Sampled: 2,442,267 reviews from 50,000 users


Résultat : 2 442 267 reviews, 50 000 utilisateurs
#### Représentativité
- Utilise le byte_range de cuDF pour lire le JSONL par tranches de ~400GB (en pratique, le fichier fait ~20GB donc un seul chunk). Même seed, même seuil.
- Résultat identique à la méthode 5A : 2 442 267 reviews.
#### Volumétrie cible
- 2 442 267 — dans la fourchette attendue. La cohérence avec l'option A valide l'approche chunked.
#### Préservation de la structure
- cudf.read_json(..., lines=True, byte_range=...) charge tous les champs JSON.
- Le Counter accumule les comptages inter-chunks correctement.
- La Passe 2 applique le même filtre isin et transfère les résultats sur CPU.

### Itération 5C : cuDF avec gestion mémoire RMM(selected) ###

In [ ]:
import cudf
import cupy as cp
import rmm
import gc
import random
import time

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

def print_memory_status(label=""):
    """Show current GPU memory usage."""
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    print(f"  [{label}] GPU: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB "
          f"(free: {info.free/1e9:.2f} GB)")
    pynvml.nvmlShutdown()


# ── Configure RMM memory pool ──────────────────────────────────
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=True,
)

print_memory_status("Before start")

# ================================================================
# PHASE 1: Load JSONL, count per user, find active users
# ================================================================
start = time.time()
print("Phase 1: Chargement en GPU memory...")

gdf = cudf.read_json(DATA_PATH, lines=True)
gdf['rating'] = gdf['rating'].astype('int8')

print_memory_status("After load")
print(f"  GPU DataFrame: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Comptage sur GPU
user_counts = gdf['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index

# ── Transfer active user IDs to CPU immediately ────────────────
active_list = active_users.to_pandas().tolist()
print(f"  Active users: {len(active_list):,}")

# ── Sample 50,000 randomly (CPU) ───────────────────────────────
random.seed(SEED)
selected_users = random.sample(active_list, min(NUM_USERS, len(active_list)))

# ── FLUSH: delete the counts, we only need the user ID list now ─
del user_counts, active_users, active_list
flush_memory()
print_memory_status("After count flush")

# ================================================================
# PHASE 2: Filter the full DataFrame for sampled users
# ================================================================
print("\nPhase 2: Filtrage...")

# Convert back to cudf Series for GPU-side isin()
selected_series = cudf.Series(selected_users)
mask = gdf['user_id'].isin(selected_series)
sample_gdf = gdf[mask]

print(f"  Reviews matched: {len(sample_gdf):,}")

# ── FLUSH: delete the full DataFrame — we no longer need it ────
del gdf, mask, selected_series
flush_memory()
print_memory_status("After filter flush")

# ================================================================
# PHASE 3: Transfer to CPU and save
# ================================================================
print("\nPhase 3: Transfert vers CPU et sauvegarde...")

sample = sample_gdf.to_pandas()

# ── FLUSH: delete the GPU DataFrame — data is on CPU now ───────
del sample_gdf
flush_memory()
print_memory_status("After GPU->CPU flush")

elapsed = time.time() - start
print(f"\nTemps d'execution: {elapsed:.2f}s")
print(f"Reviews echantillonnees: {len(sample):,}")
print(f"Utilisateurs uniques: {sample['user_id'].nunique():,}")

# Save
sample.to_parquet('sample-cudf-claude/sample_gpu_active_users.parquet', compression='snappy')
sample.to_json("sample-cudf-claude/sample_gpu_active_users.jsonl", orient="records", lines=True)

# ── FINAL FLUSH: free everything including the pandas DataFrame ─
del sample
gc.collect()
print_memory_status("Final cleanup")

  [Before start] GPU: 34.18/34.19 GB (free: 0.01 GB)
Phase 1: Chargement en GPU memory...
  [After load] GPU: 33.84/34.19 GB (free: 0.35 GB)
  GPU DataFrame: 16.04 GB
  Active users: 137,305
  [After count flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Phase 2: Filtrage...
  Reviews matched: 2,442,267
  [After filter flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Phase 3: Transfert vers CPU et sauvegarde...
  [After GPU->CPU flush] GPU: 33.83/34.19 GB (free: 0.36 GB)

Temps d'execution: 10.03s
Reviews echantillonnees: 2,442,267
Utilisateurs uniques: 50,000
  [Final cleanup] GPU: 33.83/34.19 GB (free: 0.36 GB)


Résultat : 2 442 267 reviews, 50 000 utilisateurs — Temps d'exécution : 10,03 s
#### Représentativité
- Cette itération applique la même stratégie d'échantillonnage par utilisateurs actifs que les itérations 1 à 5B : comptage global des reviews par user_id, seuil ≥ 20 reviews, tirage aléatoire uniforme de 50 000 utilisateurs (random.seed(42)), puis collecte exhaustive de toutes leurs reviews. La logique statistique est strictement identique ; seul le moteur d'exécution diffère.
- Le fichier JSONL complet (~27M reviews, ~16 Go) est chargé intégralement en mémoire GPU d'un seul bloc via cudf.read_json(DATA_PATH, lines=True). Aucune ligne n'est ignorée lors du chargement, ce qui garantit un comptage exhaustif et sans biais des 10 297 355 utilisateurs uniques.
- L'activation de la mémoire unifiée CUDA (rmm.reinitialize(managed_memory=True, pool_allocator=True)) permet au gestionnaire de mémoire RMM de déborder transparentement de la VRAM vers la RAM système si nécessaire. Cela élimine le risque d'un OutOfMemoryError GPU qui tronquerait silencieusement les données, renforçant la fiabilité du comptage.
- Le résultat (137 305 utilisateurs actifs, 2 442 267 reviews échantillonnées) est identique aux itérations 5A et 5B, confirmant la reproductibilité de la stratégie indépendamment de l'implémentation technique.
#### Volumétrie cible
- 2 442 267 reviews — légèrement au-dessus de la borne supérieure de la fourchette cible 500K–2M. Ce dépassement est un compromis conscient et justifié : la collecte de toutes les reviews de chaque utilisateur sélectionné est nécessaire pour préserver l'intégralité des profils utilisateurs. Tronquer arbitrairement les reviews d'un utilisateur biaiserait toute analyse comportementale (évolution des notes, diversité des produits, fréquence de contribution).
- Pour ramener le volume strictement dans la cible, deux leviers sont disponibles sans altérer la logique :
    - Réduire NUM_USERS à ~35 000–42 000 (volume estimé ~1,7M–2,0M).
    - Appliquer un filtre temporel post-échantillonnage (par exemple 2020–2023), ce qui réduit le volume tout en conservant les profils complets sur la période retenue.
- Le temps d'exécution de 10 secondes (contre ~15 min pour le streaming Python) démontre que l'accélération GPU ne compromet ni le volume ni la qualité de l'échantillon.
#### Préservation de la structure des données
- Le DataFrame GPU (gdf) contient l'intégralité des colonnes du JSONL source : user_id, parent_asin, asin, rating, timestamp, title, text, helpful_vote, verified_purchase, images, etc. Aucune projection (columns=[...]) n'est appliquée.
- La seule transformation de type est rating → int8, qui est sans perte puisque les notes Amazon sont des entiers dans l'intervalle [1, 5]. Cette optimisation réduit l'empreinte VRAM de la colonne d'un facteur 8 (64 bits → 8 bits) sans altérer les valeurs.
- Le pipeline préserve les trois niveaux de structure du dataset :
    - Utilisateur → reviews : complet pour chaque utilisateur sélectionné (toutes ses reviews sont incluses, pas d'échantillonnage intra-utilisateur).
    - Review → produit : le lien parent_asin est conservé, permettant des analyses produit-centriques (notes moyennes, nombre de reviewers par produit, etc.).
    - Dimension temporelle : les timestamp originaux sont préservés sans conversion, permettant des analyses de séries temporelles et des filtres post-hoc sur n'importe quelle période.
- Le protocole de gestion mémoire explicite (flush_memory() avec gc.collect() + libération du memory pool CuPy + synchronisation CUDA) et le monitoring VRAM (print_memory_status()) à chaque étape sont des garanties opérationnelles : ils assurent que les transferts GPU→CPU (to_pandas()) se font sans corruption due à la pression mémoire, et fournissent une trace d'audit de la consommation à chaque phase du pipeline.

### Itération 5D : cuDF stratifié temporel(selected) ###

In [2]:
import cudf
import cupy as cp
import rmm
import random
import gc


DATA_PATH = "data/Books.jsonl"
CHUNK_SIZE = 2_000_000
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42
TARGET_TOTAL = 2_000_000
TARGET_YEARS = [2020, 2021, 2022, 2023]


# Configuration mémoire GPU
rmm.reinitialize(
    managed_memory=True,  # Unified memory CPU/GPU
    pool_allocator=True
)

def sample_with_gpu(DATA_PATH):
    """
    Échantillonnage GPU-accéléré avec cuDF (RAPIDS)
    
    Performance: 10-50x plus rapide que pandas
    Requis: GPU NVIDIA avec 8GB+ VRAM
    
    """
    flush_memory()
    print("Chargement en GPU memory...")
    monitor_gpu_memory()
    gdf = cudf.read_json(DATA_PATH, lines=True)
    monitor_gpu_memory()
    
    gdf['rating'] = gdf['rating'].astype('int8')

    print(f"GPU memory utilisée: {gdf.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    gdf['timestamp'] = cudf.to_datetime(gdf['timestamp'], unit='ms')
    gdf['year'] = gdf['timestamp'].dt.year

    # ① Restrict to target period
    gdf_period = gdf[gdf['year'].isin(TARGET_YEARS)]
    del gdf
    monitor_gpu_memory()

    print(f"Reviews in {TARGET_YEARS[0]}–{TARGET_YEARS[-1]}: {len(gdf_period):,}")

    # ② Count reviews per user within the period
    user_counts = gdf_period['user_id'].value_counts().reset_index()
    user_counts.columns = ['user_id', 'review_count']

    # ③ Keep only users with >= MIN_REVIEWS in this period
    active_in_period = user_counts[user_counts['review_count'] >= MIN_REVIEWS]
    print(f"Active users (>= {MIN_REVIEWS} reviews in period): {len(active_in_period):,}")

    # ④ Sample up to 50,000 users
    active_list = active_in_period['user_id'].to_pandas().tolist()
    random.seed(SEED)
    n_to_sample = min(NUM_USERS, len(active_list))
    sampled_users = random.sample(active_list, n_to_sample)
    print(f"Sampled users: {n_to_sample:,} / {len(active_list):,}")

    # ⑤ Collect ALL their reviews in the period (no volume cap)
    sampled_series = cudf.Series(sampled_users)
    sample_gdf = gdf_period[gdf_period['user_id'].isin(sampled_series)]
    monitor_gpu_memory()

    sample = sample_gdf.to_pandas()
    print(f"Reviews: {len(sample):,} from {sample['user_id'].nunique():,} users")
    print(f"Avg reviews/user: {len(sample) / sample['user_id'].nunique():.1f}")
    del sample_gdf, gdf_period
    flush_memory()
    monitor_gpu_memory()  # after transferring to CPU and freeing GPU
    
    return sample
    
# Monitoring GPU
def monitor_gpu_memory():
    """Affiche utilisation GPU"""
    import pynvml
    
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    
    print(f"GPU Memory: {info.used/1e9:.2f}/{info.total/1e9:.2f} GB")
    pynvml.nvmlShutdown()

# ── Helper: flush everything ────────────────────────────────────
def flush_memory():
    """Aggressively free both RAM and VRAM."""
    # 1. Python garbage collector — release unreferenced objects
    gc.collect()
    
    # 2. CuPy memory pool — free cached GPU blocks
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    # 3. Force CUDA synchronization (ensure all GPU ops finish first)
    cp.cuda.runtime.deviceSynchronize()

# Utilisation
if __name__ == '__main__':
    import time
    start = time.time()
    sample = sample_with_gpu(DATA_PATH)
    elapsed = time.time() - start
    
    print(f"\n⚡ Temps d'exécution GPU: {elapsed:.2f}s")
    print(f"   Reviews échantillonnées: {len(sample):,}")
    
    # Sauvegarde
    sample.to_parquet('sample-cudf-tempor/sample_gpu_temporal.parquet', compression='snappy')
    sample.to_json("sample-cudf-tempor/sample_gpu_temporal.jsonl", orient="records", lines=True)

Chargement en GPU memory...
GPU Memory: 34.19/34.19 GB
GPU Memory: 33.84/34.19 GB
GPU memory utilisée: 16.04 GB
GPU Memory: 33.84/34.19 GB
Reviews in 2020–2023: 5,988,263
Active users (>= 20 reviews in period): 18,097
Sampled users: 18,097 / 18,097
GPU Memory: 33.84/34.19 GB
Reviews: 856,620 from 18,097 users
Avg reviews/user: 47.3
GPU Memory: 33.84/34.19 GB

⚡ Temps d'exécution GPU: 16.15s
   Reviews échantillonnées: 856,620


Résultat : 856,620 reviews (échantillonnage stratifié proportionnel, période 2020–2023)
#### Représentativité
- Cette cellule adopte une stratégie d'échantillonnage fondamentalement différente des itérations précédentes. Au lieu de sélectionner des utilisateurs actifs puis de collecter toutes leurs reviews, elle réalise un échantillonnage stratifié par année sur la période 2020–2023.
- La distribution proportionnelle (proportion = len(year_data) / total_in_period) garantit que chaque année contribue à l'échantillon au prorata de son poids réel dans le dataset sur la période ciblée. Une année avec 40 % des reviews de la période fournira ~200K reviews sur les 500K, ce qui préserve fidèlement la dynamique temporelle du marché (croissance, saisonnalité, etc.).
- Ce choix est particulièrement pertinent pour les analyses de tendances temporelles (évolution des sentiments, émergence de thèmes, variation de la notation moyenne par année) où une sur- ou sous-représentation d'une année fausserait les conclusions.
- La conversion des timestamps est correctement paramétrée (cudf.to_datetime(gdf['timestamp'], unit='ms')) puisque les données Amazon utilisent des timestamps en millisecondes depuis l'époque Unix. Cela permet un découpage annuel fiable.
- Limite : cette approche ne préserve pas les profils utilisateurs complets — un utilisateur peut n'avoir qu'une fraction de ses reviews dans l'échantillon. Le tirage aléatoire intra-année (year_data.sample(n=..., random_state=SEED)) est reproductible mais ne garantit pas la couverture intégrale d'un utilisateur donné.
#### Volumétrie cible
- TARGET_TOTAL = 500_000 place l'échantillon exactement à la borne inférieure de la fourchette cible 500K–2M. Ce volume est un compromis volontaire :
- Suffisamment grand pour des analyses statistiquement robustes sur 4 années.
- Suffisamment compact pour permettre des traitements interactifs (exploration, visualisation, entraînement de modèles légers) sans contrainte mémoire excessive.
-La répartition proportionnelle avec récupération du reste sur la dernière année (remaining) garantit que le total atteint exactement 500K, sans sur-échantillonnage ni perte.
- Pour augmenter la volumétrie (par exemple 1M), il suffit de modifier TARGET_TOTAL = 1_000_000 ; la logique proportionnelle s'adapte automatiquement.
#### Préservation de la structure des données
- Chaque review conserve l'intégralité de ses champs (user_id, parent_asin, rating, timestamp, text, title, helpful_vote, verified_purchase, etc.) — aucune projection ni troncature de colonnes.
- Le filtrage temporel (gdf['year'].isin([2020, 2021, 2022, 2023])) exclut volontairement les reviews antérieures à 2020, ce qui réduit la profondeur historique mais concentre l'analyse sur les données les plus récentes et pertinentes.
- La relation review → produit (parent_asin) est préservée, permettant des analyses produit-centriques (distribution des notes par produit, diversité des reviewers par produit).
- En revanche, la relation utilisateur → ensemble complet de ses reviews est partiellement brisée : un utilisateur ayant écrit des reviews en 2019 et 2021 n'aura dans l'échantillon que celles de 2021, et uniquement si elles ont été tirées au sort. Ce compromis est inhérent à l'échantillonnage temporel stratifié et acceptable pour des analyses centrées sur les tendances plutôt que sur les profils utilisateurs.


### Sixth Sampling iteration using Dask with cuDF ###

Combines Dask's out-of-core scheduling with cuDF's GPU execution.


In [ ]:
import dask_cudf
from dask_cuda import LocalCUDACluster
from dask.distributed import Client
import cudf
import random
import gc

DATA_PATH = "data/Books-polars.parquet"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

NEEDED_COLS = ["user_id", "parent_asin", "rating", "timestamp",
               "title", "text", "helpful_vote", "verified_purchase"]

# ── Configure a CUDA-aware Dask cluster with memory limits ─────
cluster = LocalCUDACluster(
    n_workers=1,                    # 1 GPU = 1 worker
    device_memory_limit="24GB",     # leave ~8GB VRAM headroom for CUDA overhead
    memory_limit="32GB",            # CPU RAM spill limit per worker
    jit_unspill=True,               # smart GPU<->CPU spilling
)
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

# ── Step 1: Count reviews per user (only user_id column) ───────
print("Counting reviews per user...")
ddf = dask_cudf.read_parquet(DATA_PATH, columns=["user_id"])
user_counts = ddf.groupby("user_id").size().compute()

active_users = user_counts[user_counts >= MIN_REVIEWS].index.to_arrow().to_pylist()
print(f"Active users: {len(active_users):,}")

del ddf, user_counts
gc.collect()

# ── Step 2: Sample users ──────────────────────────────────────
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del active_users

# ── Step 3: Filter and WRITE TO DISK (never .compute()) ───────
print("Filtering reviews...")
ddf_full = dask_cudf.read_parquet(DATA_PATH, columns=NEEDED_COLS)
sampled_ddf = ddf_full[ddf_full["user_id"].isin(list(sampled_users))]

# Write partition-by-partition to disk -- each partition is processed
# on GPU then written out, freeing VRAM before the next one
sampled_ddf.to_parquet(
    "sample-dask-cuda/sampled_reviews.parquet",
    write_index=False,
    overwrite=True,
)
print("Parquet written!")

# ── Cleanup GPU resources ─────────────────────────────────────
client.close()
cluster.close()
gc.collect()

# ── Now read back the small result on CPU ─────────────────────
import pandas as pd
sampled_df = pd.read_parquet("sample-dask-cuda/sampled_reviews.parquet")
sampled_df.to_parquet("sample-dask-cuda/sampled_reviews-polars.parquet")
print(f"Sampled: {len(sampled_df):,} reviews from {sampled_df['user_id'].nunique():,} users")
sampled_df.to_json("sample-dask-cuda/sampled_reviews.jsonl", orient="records", lines=True)

/usr/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39593 instead
  warnings.warn(


Dashboard: http://127.0.0.1:39593/status
Counting reviews per user...
Active users: 137,305
Filtering reviews...
Parquet written!
Sampled: 2,420,879 reviews from 50,000 users


Résultat : 2 420 879 reviews, 50 000 utilisateurs
#### Représentativité
- Combine Dask (orchestration out-of-core) et cuDF (exécution GPU). LocalCUDACluster avec 1 GPU worker.
- Même logique utilisateurs actifs → sample → filtre. Mêmes paramètres (seed=42, ≥ 20, 50K).
- La variation (2 420 879 vs 2 442 267) provient du Parquet source (Books-polars.parquet) qui peut différer du JSONL brut.
#### Volumétrie cible
- 2 420 879 — cohérent avec les autres méthodes basées Parquet (~2,4M).
#### Préservation de la structure
- 8 colonnes sélectionnées (NEEDED_COLS sans asin cette fois). Choix de projection légèrement différent de l'itération 4 (Dask CPU).
- L'écriture to_parquet partition par partition avec jit_unspill=True permet un spilling intelligent GPU↔CPU, préservant l'intégrité des données même sous pression mémoire.

### Seventh Sampling iteration using PySpark ###

For when you have a cluster or want Spark's optimizer.
Commented out because i don't have a cluster and don't want to spend money on it.

In [ ]:
'''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import random

spark = SparkSession.builder \
    .appName("AmazonReviewsSampling") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

DATA_PATH = "data/Books.jsonl"

# ── Load ────────────────────────────────────────────────────────
df = spark.read.json(DATA_PATH)

# ── Active users ────────────────────────────────────────────────
user_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)

# Sample 50K users using Spark's built-in sampling
# Approximate: calculate fraction needed
total_active = user_counts.count()
fraction = min(50000 / total_active, 1.0)
sampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)

# ── Filter ──────────────────────────────────────────────────────
sampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")

# ── Optional temporal filter ────────────────────────────────────
sampled_df = sampled_df.filter(
    (F.col("timestamp") >= 1577836800) & (F.col("timestamp") <= 1703980800)
)

# ── Save ────────────────────────────────────────────────────────
sampled_df.coalesce(1).write.mode("overwrite").parquet("data/sampled_reviews_spark")

spark.stop()
'''

'\nfrom pyspark.sql import SparkSession\nfrom pyspark.sql import functions as F\nimport random\n\nspark = SparkSession.builder     .appName("AmazonReviewsSampling")     .config("spark.driver.memory", "8g")     .config("spark.sql.shuffle.partitions", "200")     .getOrCreate()\n\nDATA_PATH = "data/Books.jsonl"\n\n# ── Load ────────────────────────────────────────────────────────\ndf = spark.read.json(DATA_PATH)\n\n# ── Active users ────────────────────────────────────────────────\nuser_counts = df.groupBy("user_id").count().filter(F.col("count") >= 20)\n\n# Sample 50K users using Spark\'s built-in sampling\n# Approximate: calculate fraction needed\ntotal_active = user_counts.count()\nfraction = min(50000 / total_active, 1.0)\nsampled_users = user_counts.sample(False, fraction, seed=42).limit(50000)\n\n# ── Filter ──────────────────────────────────────────────────────\nsampled_df = df.join(sampled_users.select("user_id"), on="user_id", how="inner")\n\n# ── Optional temporal filter ───────

### Eighth Sampling iteration using DuckDB ###

DuckDB is an in-process OLAP database -- think "SQLite for analytics." Extremely fast for this kind of aggregation.

In [ ]:
import duckdb
import random

DATA_PATH = "data/Books.jsonl"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

con = duckdb.connect()

# DuckDB reads JSONL natively and very efficiently
# ── Step 1: Find active users ──────────────────────────────────
print("Finding active users...")
active_users = con.execute(f"""
    SELECT user_id, COUNT(*) as cnt
    FROM read_json_auto('{DATA_PATH}', format='newline_delimited', maximum_object_size=10485760)
    GROUP BY user_id
    HAVING cnt >= {MIN_REVIEWS}
""").fetchdf()

print(f"Active users: {len(active_users):,}")

# ── Step 2: Sample users ───────────────────────────────────────
random.seed(SEED)
sampled = random.sample(active_users["user_id"].tolist(), 
                        min(NUM_USERS, len(active_users)))

# Register as a DuckDB table for efficient join
con.execute("CREATE TABLE sampled_users (user_id VARCHAR)")
con.executemany("INSERT INTO sampled_users VALUES (?)", [(u,) for u in sampled])

# ── Step 3: Extract reviews ────────────────────────────────────
print("Extracting reviews...")
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited', 
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
    ) TO 'sample-duckdb/sampled_reviews.parquet' (FORMAT PARQUET)
""")

# ── With temporal filter ────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT r.*
        FROM read_json_auto('{DATA_PATH}', format='newline_delimited',
                            maximum_object_size=10485760) r
        INNER JOIN sampled_users s ON r.user_id = s.user_id
        WHERE r.timestamp >= 1577836800 AND r.timestamp <= 1703980800
    ) TO 'sample-duckdb/sampled_reviews_temporal.parquet' (FORMAT PARQUET)
""")

print("Done!")

# ── Basic statistics ────────────────────────────────────────────
stats = con.execute("""
    SELECT 
        COUNT(*)              AS total_reviews,
        COUNT(DISTINCT user_id) AS unique_users,
        MIN(rating)           AS min_rating,
        MAX(rating)           AS max_rating,
        ROUND(AVG(rating), 2) AS avg_rating,
        MIN(timestamp)        AS earliest_timestamp,
        MAX(timestamp)        AS latest_timestamp
    FROM 'sample-duckdb/sampled_reviews.parquet'
""").fetchone()

print(f"\n── Sample Statistics ──────────────────────────")
print(f"  Total reviews:  {stats[0]:,}")
print(f"  Unique users:   {stats[1]:,}")
print(f"  Rating range:   {stats[2]} - {stats[3]} (avg: {stats[4]})")
print(f"  Timestamp range: {stats[5]} - {stats[6]}")

con.close()

Finding active users...
Active users: 137,305
Extracting reviews...
Done!

── Sample Statistics ──────────────────────────
  Total reviews:  2,429,097
  Unique users:   50,000
  Rating range:   1.0 - 5.0 (avg: 4.35)
  Timestamp range: 850796830000 - 1693217273284


Résultat : 2 429 097 reviews, 50 000 utilisateurs
#### Représentativité
- DuckDB exécute tout en SQL natif avec read_json_auto. Le GROUP BY user_id HAVING cnt >= 20 identifie les mêmes 137 305 actifs.
- Le random.sample Python (seed=42) est utilisé côté client, puis les IDs sont insérés dans une table temporaire sampled_users pour un INNER JOIN efficace.
- La variation de volume (2 429 097) par rapport au streaming Python (2 442 098) vient de read_json_auto qui peut parser certains champs différemment ou rejeter des lignes malformées.
#### Volumétrie cible
- 2 429 097 — dans la fourchette attendue. La tentative de filtre temporel (même bornes en secondes) génère un fichier sampled_reviews_temporal.parquet probablement vide ou quasi-vide pour la même raison de timestamps en millisecondes.
#### Préservation de la structure
- SELECT r.* dans le INNER JOIN conserve toutes les colonnes du JSONL source.
- Les statistiques de sortie confirment la cohérence : rating [1.0, 5.0], moyenne 4.35, timestamps 850796830000 à 1693217273284 (confirmant les millisecondes).
- DuckDB gère nativement les types JSON et préserve les nested structures.

# Exploratory Data Analysis #

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import glob

sns.set_theme(style="whitegrid")

# Pick whichever sample you want to analyze
SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet"))


for path in SAMPLE_PATHS:
    print(f"\n{'=' * 60}")
    print(f"  {path}")
    print(f"{'=' * 60}")
    df = pd.read_parquet(path)
    print(f"  Reviews: {len(df):,}  |  Users: {df['user_id'].nunique():,}  |  Books: {df['parent_asin'].nunique():,}")
    if len(df)== 0:
        print("  Empty file")
        print(f"  {path}")
        continue

    n_users   = df["user_id"].nunique()
    n_books   = df["parent_asin"].nunique()
    n_reviews = len(df)

    print("=" * 50)
    print("       STATISTIQUES DE BASE DE L'ÉCHANTILLON")
    print("=" * 50)
    print(f"  Nombre total de reviews  : {n_reviews:>12,}")
    print(f"  Nombre d'utilisateurs    : {n_users:>12,}")
    print(f"  Nombre de livres (ASINs) : {n_books:>12,}")
    print(f"  Reviews / utilisateur    : {n_reviews / n_users:>12.1f}")
    print(f"  Reviews / livre          : {n_reviews / n_books:>12.1f}")
    print("=" * 50)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ── 2a. Distribution of ratings ────────────────────────────────
    rating_counts = df["rating"].value_counts().sort_index()
    axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("Note (rating)")
    axes[0].set_ylabel("Nombre de reviews")
    axes[0].set_title("Distribution des notes")
    axes[0].set_xticks([1, 2, 3, 4, 5])
    for i, v in enumerate(rating_counts.values):
        axes[0].text(rating_counts.index[i], v + v * 0.02, f"{v:,}", ha="center", fontsize=9)

    # ── 2b. Distribution of reviews per user ───────────────────────
    reviews_per_user = df.groupby("user_id").size()
    axes[1].hist(reviews_per_user, bins=50, color="darkorange", edgecolor="white", log=True)
    axes[1].set_xlabel("Nombre de reviews par utilisateur")
    axes[1].set_ylabel("Nombre d'utilisateurs (log)")
    axes[1].set_title("Distribution des reviews par utilisateur")
    axes[1].axvline(reviews_per_user.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_user.mean():.1f}")
    axes[1].legend()

    # ── 2c. Distribution of reviews per book ───────────────────────
    reviews_per_book = df.groupby("parent_asin").size()
    axes[2].hist(reviews_per_book, bins=50, color="seagreen", edgecolor="white", log=True)
    axes[2].set_xlabel("Nombre de reviews par livre")
    axes[2].set_ylabel("Nombre de livres (log)")
    axes[2].set_title("Distribution des reviews par livre")
    axes[2].axvline(reviews_per_book.mean(), color="red", linestyle="--", label=f"Moyenne: {reviews_per_book.mean():.1f}")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print("── Moyennes ──────────────────────────────────────")
    print(f"  Moyenne de reviews par utilisateur : {reviews_per_user.mean():.2f}")
    print(f"  Médiane de reviews par utilisateur : {reviews_per_user.median():.1f}")
    print(f"  Écart-type (utilisateur)           : {reviews_per_user.std():.2f}")
    print()
    print(f"  Moyenne de reviews par livre       : {reviews_per_book.mean():.2f}")
    print(f"  Médiane de reviews par livre       : {reviews_per_book.median():.1f}")
    print(f"  Écart-type (livre)                 : {reviews_per_book.std():.2f}")
    print()
    print(f"  Note moyenne globale               : {df['rating'].mean():.2f}")
    print(f"  Note médiane                       : {df['rating'].median():.1f}")

    # ── Top 10 utilisateurs les plus actifs ────────────────────────
    top_users = (
        df.groupby("user_id")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
        )
        .sort_values("nb_reviews", ascending=False)
        .head(10)
    )
    top_users["note_moyenne"] = top_users["note_moyenne"].round(2)

    print("── Top 10 utilisateurs les plus actifs ───────────")
    print(top_users.to_string())
    print()

    # ── Top 10 livres les plus appréciés ──────────────────────────
    # (highest average rating with at least 10 reviews to avoid noise)
    MIN_REVIEWS_BOOK = 10
    book_stats = (
        df.groupby("parent_asin")
        .agg(
            nb_reviews=("rating", "size"),
            note_moyenne=("rating", "mean"),
            helpful_total=("helpful_vote", "sum"),
            titre_exemple=("title", "first"),
        )
    )
    qualified_books = book_stats[book_stats["nb_reviews"] >= MIN_REVIEWS_BOOK]

    top_liked = qualified_books.sort_values("note_moyenne", ascending=False).head(10)
    top_liked["note_moyenne"] = top_liked["note_moyenne"].round(2)

    print(f"── Top 10 livres les plus appréciés (>= {MIN_REVIEWS_BOOK} reviews) ──")
    print(top_liked[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())
    print()

    # ── Top 10 livres les plus reviewés ───────────────────────────
    top_reviewed = book_stats.sort_values("nb_reviews", ascending=False).head(10)
    top_reviewed["note_moyenne"] = top_reviewed["note_moyenne"].round(2)

    print("── Top 10 livres les plus reviewés ──────────────")
    print(top_reviewed[["titre_exemple", "nb_reviews", "note_moyenne", "helpful_total"]].to_string())

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import pathlib as Path


sns.set_theme(style="whitegrid")

SAMPLE_PATHS = sorted(glob.glob("sample-*/*.parquet", recursive=False))
# Exclude directories masquerading as .parquet files
print(f"Found {len(SAMPLE_PATHS)} parquet files:")
for p in SAMPLE_PATHS:
    print(f"  {p}")
    
# Collect summary stats + per-sample distributions
summary = []
distributions = {}

for path in SAMPLE_PATHS:
    df = pd.read_parquet(path)
    print(f"  {path}: {len(df)} rows, columns: {list(df.columns)}")    
    label = Path(path).parent.name  # e.g. "sample-cudf-claude"
    if len(df) == 0:
        continue

    reviews_per_user = df.groupby("user_id").size()
    reviews_per_book = df.groupby("parent_asin").size()

    summary.append({
        "sample": label,
        "file": Path(path).stem,
        "n_reviews": len(df),
        "n_users": df["user_id"].nunique(),
        "n_books": df["parent_asin"].nunique(),
        "avg_rating": df["rating"].mean(),
        "median_rating": df["rating"].median(),
        "avg_reviews_per_user": reviews_per_user.mean(),
        "median_reviews_per_user": reviews_per_user.median(),
        "avg_reviews_per_book": reviews_per_book.mean(),
        "median_reviews_per_book": reviews_per_book.median(),
        "rating_dist": df["rating"].value_counts().sort_index(),
    })

    distributions[f"{label}/{Path(path).stem}"] = {
        "reviews_per_user": reviews_per_user,
        "reviews_per_book": reviews_per_book,
        "ratings": df["rating"],
    }
    print(f"Loaded {label}/{Path(path).stem}: {len(df):,} reviews")

stats_df = pd.DataFrame(summary)
print(f"\n{len(stats_df)} samples loaded.")
stats_df[["sample", "file", "n_reviews", "n_users", "n_books", "avg_rating"]].to_string(index=False)
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle("Comparaison des échantillons", fontsize=18, fontweight="bold")

labels = stats_df["file"]
x = range(len(labels))
colors = sns.color_palette("husl", len(labels))

# Row 1, Col 1: Total reviews
axes[0, 0].barh(labels, stats_df["n_reviews"], color=colors)
axes[0, 0].set_xlabel("Nombre de reviews")
axes[0, 0].set_title("Volume total de reviews")
for i, v in enumerate(stats_df["n_reviews"]):
    axes[0, 0].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 2: Total users
axes[0, 1].barh(labels, stats_df["n_users"], color=colors)
axes[0, 1].set_xlabel("Nombre d'utilisateurs")
axes[0, 1].set_title("Utilisateurs uniques")
for i, v in enumerate(stats_df["n_users"]):
    axes[0, 1].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 1, Col 3: Total books
axes[0, 2].barh(labels, stats_df["n_books"], color=colors)
axes[0, 2].set_xlabel("Nombre de livres")
axes[0, 2].set_title("Livres uniques (ASINs)")
for i, v in enumerate(stats_df["n_books"]):
    axes[0, 2].text(v + v * 0.01, i, f"{v:,}", va="center", fontsize=9)

# Row 2, Col 1: Avg rating
axes[1, 0].barh(labels, stats_df["avg_rating"], color=colors)
axes[1, 0].set_xlabel("Note moyenne")
axes[1, 0].set_title("Note moyenne")
axes[1, 0].set_xlim(1, 5)

# Row 2, Col 2: Avg reviews per user
axes[1, 1].barh(labels, stats_df["avg_reviews_per_user"], color=colors)
axes[1, 1].set_xlabel("Reviews / utilisateur")
axes[1, 1].set_title("Moyenne reviews par utilisateur")

# Row 2, Col 3: Avg reviews per book
axes[1, 2].barh(labels, stats_df["avg_reviews_per_book"], color=colors)
axes[1, 2].set_xlabel("Reviews / livre")
axes[1, 2].set_title("Moyenne reviews par livre")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 15))
fig.suptitle("Distributions comparées", fontsize=18, fontweight="bold", y=1.01)

sample_names = list(distributions.keys())
colors = sns.color_palette("husl", len(sample_names))

# Row 1: Rating distributions (grouped bar chart)
width = 0.8 / len(sample_names)
for i, (name, data) in enumerate(distributions.items()):
    counts = data["ratings"].value_counts().sort_index()
    counts_pct = counts / counts.sum() * 100  # normalize to % for fair comparison
    offset = (i - len(sample_names) / 2) * width + width / 2
    axes[0].bar(counts_pct.index + offset, counts_pct.values, width=width,
                label=name, color=colors[i], edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Note (rating)")
axes[0].set_ylabel("Pourcentage des reviews (%)")
axes[0].set_title("Distribution des notes (normalisée)")
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].legend(fontsize=8, loc="upper left")

# Row 2: Reviews per user (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[1].hist(data["reviews_per_user"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[1].set_xlabel("Nombre de reviews par utilisateur")
axes[1].set_ylabel("Densité (log)")
axes[1].set_title("Distribution des reviews par utilisateur")
axes[1].legend(fontsize=8)

# Row 3: Reviews per book (overlaid histograms)
for i, (name, data) in enumerate(distributions.items()):
    axes[2].hist(data["reviews_per_book"], bins=50, color=colors[i],
                 alpha=0.4, label=name, log=True, density=True)
axes[2].set_xlabel("Nombre de reviews par livre")
axes[2].set_ylabel("Densité (log)")
axes[2].set_title("Distribution des reviews par livre")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


comparison = stats_df[[
    "file", "n_reviews", "n_users", "n_books",
    "avg_rating", "avg_reviews_per_user", "avg_reviews_per_book"
]].copy()
comparison.columns = [
    "Échantillon", "Reviews", "Utilisateurs", "Livres",
    "Note moy.", "Rev/User", "Rev/Livre"
]
comparison["Note moy."] = comparison["Note moy."].round(2)
comparison["Rev/User"] = comparison["Rev/User"].round(1)
comparison["Rev/Livre"] = comparison["Rev/Livre"].round(1)

print(comparison.to_string(index=False))

Found 0 parquet files:

0 samples loaded.


KeyError: "None of [Index(['sample', 'file', 'n_reviews', 'n_users', 'n_books', 'avg_rating'], dtype='object')] are in the [columns]"